# Germination

## Objectives {.unnumbered}

- Review the hydrothermal time (HTT) model framework for seed germination based on @Grundy2000 and @Alvarado2002.
- Implement the HTT model in pure Julia and using Cropbox.jl.
- Visualize model predictions for temperature and water potential effects on germination.
- Explore other population-based approaches to modeling germination.

## Readings {.unnumbered}

1. (**required**) Grundy AC, Phelps K, Reader RJ, Burston S (2000) Modelling the germination of *Stellaria media* using the concept of hydrothermal time. New Phytologist 148(3), 433-444 (<https://doi.org/10.1046/j.1469-8137.2000.00778.x>)

2. (recommended) Alvarado V, Bradford KJ (2002) A hydrothermal time model explains the cardinal temperatures for seed germination. Plant, Cell & Environment 25:1061-1069 (<https://doi.org/10.1046/j.1365-3040.2002.00894.x>)

3. (recommended) Thornley JHM (1986) A germination model: Responses to time and temperature. Journal of Theoretical Biology 123(4), 481-492 (<https://doi.org/10.1016/S0022-5193(86)80215-6>)

4.  (recommended) Finch-Savage WE, Leubner-Metzger G (2006) Seed dormancy and the control of germination. New Phytologist 171(3), 501-523 (<https://doi.org/10.1111/j.1469-8137.2006.01787.x>)

5. (optional) Onofri A, Mesgaran MB, Ritz C (2022) A unified framework for the analysis of germination, emergence, and other time-to-event data in weed science. Weed Science 70(3), 259-271 (<https://doi.org/10.1017/wsc.2022.8>)

## Questions {.unnumbered}

1. How would dormancy-breaking treatments (e.g., stratification) affect the model parameters $m_L$ and $\sigma_L$?

2. The Grundy model assumes a constant $\Psi_b$ for the population. How might introducing variation in $\Psi_b$ (similar to temperature thresholds) improve predictions?

3. Compare the hydrothermal time approach with the temperature driven compartment germination model by Thornley (1986). What are the advantages and disadvantages of each approach? 

4. How could this model be extended to predict germination under fluctuating temperature and water potential conditions?

### Keywords {.unnumbered}

hydrothermal time, thermal time, hydrotime, base temperature, base water potential, cardinal temperatures, population-based model, probit analysis, seed germination, time-to-event modeling 

## Introduction

Seed germination is a critical life stage for plants as they begin the growth. Germination is often modeled as a popultion process rather than as an individual event because of the inherent genetic and environmental variabilities associated with seed populations.

We will examine a few case studies of germination modeling, focusing on the hydrothermal time model proposed by Grundy et al. (2000) for *Stellaria media* (common chickweed). This paper illustrates a simple but robust hydrothermal time modeling (HTT) framework that integrates temperature and water potential to determine the timing and success of seed germination by combining thermal time and hydrotime concepts. A similar case study by Alvarado and Bradford (2002) extends the hydrothermal time model to operate based on cardinal temperatures. While somewhat more complex, a compartmental germination model by Thornley (1986) provides additional insights into the rate limiting  processes underlying germination. A more statistical approach to germination is presented by Onofri et al. (2022) which proposes a time-to-event modeling framework.

### The hydrothermal time concept

As we saw previously with other processes, the rate of germination increases with temperature above a base temperature ($T_b$) and below an optimal temperature ($T_o$). This relationship is captured by the familiar **growing degree day (GDD)** or thermal time concept:

$$
\theta_T = (T - T_b) \cdot t_g
$$ {#eq-seed-thermal-time}

where $\theta_T$ is the thermal time requirement (°C·d), $T$ is the temperature (°C), $T_b$ is the base temperature (°C), and $t_g$ is the time to germination (d).

Similarly, germination rate depends on water availability characterized by water potential ($\Psi$):

$$
\theta_H = (\Psi - \Psi_b(g)) \cdot t_g
$$ {#eq-hydrotime}

where $\theta_H$ is the hydrotime constant (MPa·d), $\Psi$ is the water potential (MPa), and $\Psi_b(g)$ is the base water potential for the $g$-th percentile of the population (MPa).

Combining these two concepts, Grundy et al. (2000) applied these concepts into a comprehensive **hydrothermal time model** for predicting germination of *Stellaria media* (common chickweed):

$$
\theta_{HT} = (\Psi - \Psi_b)(T - T_b) \cdot t_g
$$ {#eq-htt}

This model also accounts for temperature thresholds that limit germination at both low and high temperatures through population-based distributions.

## Grundy model

Grundy AC, Phelps K, Reader RJ, Burston S (2000) Modelling the germination of Stellaria media using the concept of hydrothermal time. New Phytologist 148(3), 433-444 (<https://doi.org/10.1046/j.1469-8137.2000.00778.x>)

### Model description

The Grundy et al. (2000) model applies the hydrothermal time concept with population-based threshold distributions to predict germination timing and final germination percentage based [probability distribution functions (PDFs) and cumulative distribution functions (CDFs)](https://medium.com/@abhishekjainindore24/understanding-probability-density-functions-pdf-and-cumulative-distribution-functions-cdf-f851aaef6e4c). They first define threshold distributions for lower and upper temperature limits for germination using PDFs of normal distribution and then derive the corresponding CDFs to apply to the germination predictions.

![PDF and CDF concepts](https://miro.medium.com/v2/resize:fit:1302/format:webp/0*9OqHQqpMhBZj0qod){#fig-pdf-cdf width="75%"}


The complete model predicts final germination percentage as a function of temperature using population-based threshold distributions:

$$
p(T) = (1 - G_u) \cdot \Phi_L(T) \cdot [1 - \Phi_U(T)]
$$ {#eq-final-germ}

where:
- $G_u$ is the ungerminated fraction (seeds that never germinate)
- $\Phi_L(T) = \Phi\left(\frac{T - m_L}{\sigma_L}\right)$ is the CDF of the lower temperature threshold
- $\Phi_U(T) = \Phi\left(\frac{T - m_U}{\sigma_U}\right)$ is the CDF of the upper temperature threshold
- $m_L, \sigma_L$ are the mean and standard deviation of the lower threshold distribution
- $m_U, \sigma_U$ are the mean and standard deviation of the upper threshold distribution

#### Time to Germination

The time for the $g$-th percentile of the seed population to germinate is given by:

$$
t_g = \frac{\theta_H(g)}{(\Psi - \Psi_b)(T - T_b)}
$$ {#eq-time-to-germ}

where the hydrothermal time for percentile $g$ follows a normal distribution:

$$
\theta_H(g) = \theta_H - \sigma_H \cdot \Phi^{-1}\left(\frac{g}{100}\right)
$$ {#eq-theta-percentile}

#### Model parameters

| Parameter | Value | Unit | Description |
|-----------|-------|------|-------------|
| $G_u$ | 0.0 | - | Ungerminated fraction |
| $m_L$ | 7.3 | °C | Mean lower temperature threshold |
| $\sigma_L$ | 4.09 | K | SD of lower threshold |
| $m_U$ | 25.9 | °C | Mean upper temperature threshold |
| $\sigma_U$ | 5.01 | K | SD of upper threshold |
| $T_b$ | 1.42 | °C | Base temperature |
| $\Psi_b$ | -1.33 | MPa | Base water potential |
| $\theta_H$ | 47.99 | MPa·°C·d | Hydrothermal time constant |
| $\sigma_H$ | 10.46 | MPa·°C·d | SD of hydrothermal time |

### Model implementation in `Julia`

#### Load packages and setup

In [ ]:
using Logging
Logging.disable_logging(Logging.Warn)

# Load required packages
using DataFrames
using Distributions
using CSV
using Statistics
using Printf
import Plots

# Set plot defaults
Plots.gr()
Plots.default(dpi=300, size=(600, 400), legend=:best)

# Create plot aliases to avoid name conflicts
const scatter! = Plots.scatter!
const hline! = Plots.hline!
const vline! = Plots.vline!
const savefig = Plots.savefig
const annotate! = Plots.annotate!;

#### Model parameters

In [ ]:
#| output: false

# Model parameters from Grundy et al. (2000)
struct GrundyModel
    # Final percentage germination parameters
    G_u::Float64          # Proportion ungerminated
    m_L::Float64          # Mean lower threshold (°C)
    σ_L::Float64          # SD lower threshold (°C)
    m_U::Float64          # Mean upper threshold (°C)
    σ_U::Float64          # SD upper threshold (°C)
    
    # Hydrothermal time parameters (Method 2)
    T_b::Float64          # Base temperature (°C)
    Ψ_b::Float64          # Base water potential at 50th percentile (MPa)
    θ_H::Float64          # Hydrothermal time constant (MPa °C days)
    σ_H::Float64          # SD of hydrothermal time distribution
end

# Initialize model with actual values reported in paper (page 436-438)
# G_u was set to 0 as "almost 100% of the seeds germinated under some conditions"
# Temperature thresholds from page 436: "means...were 7.3 and 25.9 °C, respectively"
# Standard deviations from page 436: "were 4.09 and 5.01 respectively"
# Hydrothermal parameters from page 438: Method 2 estimates
model = GrundyModel(
    0.0,      # G_u: set to 0 per paper (page 436: "G_u set equal to 0")
    7.3,      # m_L: mean lower threshold (°C)
    4.09,     # σ_L: SD lower threshold (°C)
    25.9,     # m_U: mean upper threshold (°C)
    5.01,     # σ_U: SD upper threshold (°C)
    1.42,     # T_b: base temperature (°C) - Method 2, page 438
    -1.33,    # Ψ_b: base water potential at G(50) (MPa) - Method 2, page 438
    47.99,    # θ_H: hydrothermal time constant (MPa °C days) - Method 2, page 438
    10.46     # σ_H: SD of hydrothermal time - page 440
)

println("Grundy Model Parameters:")
println("  Base temperature (T_b): $(model.T_b) °C")
println("  Base water potential (Ψ_b): $(model.Ψ_b) MPa")
println("  Hydrothermal time constant (θ_H): $(model.θ_H) MPa °C days")
println("  Lower threshold: $(model.m_L) ± $(model.σ_L) °C")
println("  Upper threshold: $(model.m_U) ± $(model.σ_U) °C")

#### Core model functions

In [ ]:
#| output: false

"""
    final_germination_percentage(T, model)

Calculate final percentage germination as a function of temperature.
Implements Equation 2 from Grundy et al. (2000).

Corrected formula uses multiplication (intersection of probabilities)
not subtraction as initially misread from PDF.
"""
function final_germination_percentage(T::Real, m::GrundyModel)::Float64
    # Cumulative normal distribution for lower threshold
    # Φ_L = probability that a seed's lower threshold is below T
    Φ_L = cdf(Normal(m.m_L, m.σ_L), T)
    
    # Cumulative normal distribution for upper threshold  
    # Φ_U = probability that a seed's upper threshold is below T
    Φ_U = cdf(Normal(m.m_U, m.σ_U), T)
    
    # Combined effect (Equation 2): 
    # Seeds must be both above lower threshold and below upper threshold
    # This is the intersection (multiplication) not difference (subtraction)
    # Biological logic: germination occurs if T_L < T < T_U
    # p = (1 - G_u) × P(T > T_L) × P(T < T_U)
    # p = (1 - G_u) × Φ_L × (1 - Φ_U)
    p = (1 - m.G_u) * Φ_L * (1 - Φ_U)
    
    # Convert to percentage and constrain to [0, 100]
    return max(0.0, min(100.0, p * 100))
end

"""
    θ_H_percentile(g, model)

Calculate the hydrothermal time constant for a given germination percentile.
Implements Equation 1: θH(g) = θH - σH·Φ⁻¹(g)
"""
function θ_H_percentile(g::Real, m::GrundyModel)::Float64
    return m.θ_H - m.σ_H * quantile(Normal(), g/100)
end

"""
    t_g_function(g, T, Ψ, model)

Calculate the time (in DAYS) required for a fraction g to germinate at temperature T and water potential Ψ.
Implements the rearranged equation: tg = θH(g) / ((Ψ - Ψb)(T - Tb))
Note: Returns time in DAYS since θ_H is in MPa °C days.
"""
function t_g_function(g::Real, T::Real, Ψ::Real, m::GrundyModel)::Float64
    θ_Hg = θ_H_percentile(g, m)
    denominator = (Ψ - m.Ψ_b) * (T - m.T_b)
    
    if denominator > 0
        return θ_Hg / denominator  # Returns time in DAYS
    else
        return Inf
    end
end

"""
    germination_time_course(T, Ψ, times, model)

Calculate germination percentage over time for given temperature and water potential.
"""
function germination_time_course(T::Real, Ψ::Real, times::AbstractVector{<:Real}, m::GrundyModel)::Vector{Float64}
    # Calculate time to germination for each percentile
    g_percentiles = 1:99
    t_g_values = [t_g_function(g, T, Ψ, m) for g in g_percentiles]
    
    # For each time point, count how many seeds have germinated
    results = zeros(Float64, length(times))
    for (i, t) in enumerate(times)
        count = sum(t_g_values .<= t)
        results[i] = count
    end
    
    return results
end;

#### Observed data digitized from the paper

In [ ]:
#| echo: false
#| output: false

# Observed data digitized from Figure 2 (final germination vs temperature)
obs_fig2 = DataFrame(
    temperature = [6, 10, 14, 16, 18, 20, 22, 24, 26, 30],
    germination = [40, 75, 95, 97, 95, 92, 80, 65, 50, 35],
    error = [5, 5, 3, 2, 3, 3, 5, 5, 8, 5]
)

# Observed data from Figure 3 (time courses at Ψ = -0.3 MPa)
obs_fig3 = Dict(
    6 => [(0,0), (50,5), (100,15), (150,25), (200,32), (250,35)],
    10 => [(0,0), (40,10), (80,30), (120,50), (160,65), (200,72)],
    14 => [(0,0), (20,15), (40,40), (60,65), (80,82), (100,90), (120,93)],
    18 => [(0,0), (15,20), (30,55), (45,78), (60,88), (75,93), (90,95)],
    22 => [(0,0), (20,25), (40,60), (60,80), (80,88), (100,90)],
    26 => [(0,0), (40,15), (80,35), (120,45), (160,48), (200,50)],
    30 => [(0,0), (60,8), (120,18), (180,28), (240,33), (300,35)]
)

# Observed data from Figure 5 (median germination time vs water potential)
obs_fig5 = Dict(
    14 => [(-0.3,55), (-0.5,65), (-0.7,80), (-0.9,110), (-1.1,180)],
    18 => [(-0.3,40), (-0.5,48), (-0.7,58), (-0.9,75), (-1.1,110), (-1.3,200)],
    22 => [(-0.3,35), (-0.5,42), (-0.7,52), (-0.9,65), (-1.1,95), (-1.3,160)]
)

println("Observed data digitized from paper figures")

### Figures

#### Figure 1: Temperature threshold PDFs and CDFs

This figure illustrates the population-based approach to modeling temperature limitations on germination.

##### Panel (a): Probability Density Functions

The distributions of lower ($T_L$) and upper ($T_U$) temperature thresholds in the seed population:

$$
f_L(T) = \frac{1}{\sigma_L\sqrt{2\pi}} \exp\left[-\frac{(T - m_L)^2}{2\sigma_L^2}\right]
$$

$$
f_U(T) = \frac{1}{\sigma_U\sqrt{2\pi}} \exp\left[-\frac{(T - m_U)^2}{2\sigma_U^2}\right]
$$

##### Panel (b): Cumulative Distribution Functions

- *Lower threshold ($\Phi_L$)*: Fraction of seeds capable of germinating at temperature $T$ (above their individual $T_L$)
- *Upper threshold ($1 - \Phi_U$)*: Fraction of seeds not inhibited by high temperature

##### Panel (c): Combined Effect

The final germination percentage combines both constraints:

$$
p(T) = (1 - G_u) \cdot \Phi_L(T) \cdot [1 - \Phi_U(T)]
$$

This produces the characteristic bell-shaped temperature response curve.

In [ ]:
#| echo: false

# Create temperature range
temps = range(-10, 50, length=300)

# Calculate probability densities for thresholds
lower_pdf = [pdf(Normal(model.m_L, model.σ_L), T) for T in temps]
upper_pdf = [pdf(Normal(model.m_U, model.σ_U), T) for T in temps]

# Calculate cumulative distributions
# For lower threshold: Φ_L(T) = proportion of seeds with threshold < T (can germinate)
# For upper threshold: 1 - Φ_U(T) = proportion of seeds with threshold > T (not inhibited, can germinate)
lower_cdf = [cdf(Normal(model.m_L, model.σ_L), T) * 100 for T in temps]
upper_cdf = [(1 - cdf(Normal(model.m_U, model.σ_U), T)) * 100 for T in temps]

# Calculate combined final germination
final_germ = [final_germination_percentage(T, model) for T in temps]

# Create three-panel figure
p1 = Plots.plot(temps, lower_pdf, 
    label="Lower threshold", 
    line=:dash, 
    linewidth=2,
    xlabel="", 
    ylabel="Probability density",
    title="(a) Distribution of thresholds",
    color=:blue,
    framestyle=:box)
Plots.plot!(p1, temps, upper_pdf, 
    label="Upper threshold", 
    line=:solid, 
    linewidth=2,
    color=:red)

p2 = Plots.plot(temps, lower_cdf, 
    label="Lower", 
    line=:dash, 
    linewidth=2,
    xlabel="", 
    ylabel="Percentage (%)",
    title="(b) Cumulative distributions",
    ylims=(0, 100),
    color=:blue,
    framestyle=:box)
Plots.plot!(p2, temps, upper_cdf, 
    label="Upper", 
    line=:solid, 
    linewidth=2.5,
    color=:red)

p3 = Plots.plot(temps, final_germ, 
    line=:solid, 
    linewidth=2.5,
    xlabel="Temperature (°C)", 
    ylabel="Percentage (%)",
    title="(c) Combined",
    ylims=(0, 100),
    color=:black,
    legend=false,
    framestyle=:box)

fig1 = Plots.plot(p1, p2, p3, 
    layout=(3, 1), 
    size=(600, 900),
    margin=5Plots.mm)

savefig(fig1, "grundy_fig1_temperature_range.png")
display(fig1)

println("Figure 1")

#### Figure 2: Final germination percetange vs temperature

This figure validates the model against observed data from Grundy et al. (2000). The model predicts maximum germination near 15-16°C, with declining germination at both lower and higher temperatures due to:

- *Low temperature limitation*: Seeds fail to germinate below their individual $T_L$ threshold
- *High temperature inhibition*: Seeds are inhibited above their individual $T_U$ threshold

The optimum temperature ($T_{opt}$) can be found by solving:

$$
\frac{dp(T)}{dT} = 0
$$

For the parameters from Grundy et al. (2000), $T_{opt} \approx 16°C$.

In [ ]:
#| echo: false

# Temperature range for final germination
temps_fine = 0:0.5:35

# Calculate model predictions
final_germ_model = [final_germination_percentage(T, model) for T in temps_fine]

# Plot model fit with observed data
fig2 = Plots.plot(temps_fine, final_germ_model,
    line=:solid,
    linewidth=2.5,
    xlabel="Temperature (°C)",
    ylabel="Final germination (%)",
    title="Figure 2: Final germination vs temperature",
    label="Model prediction",
    ylims=(0, 105),
    xlims=(0, 35),
    color=:blue,
    legend=:bottomleft)

# Add observed data
scatter!(fig2, obs_fig2.temperature, obs_fig2.germination,
    marker=:circle,
    markersize=6,
    markerstrokewidth=1.5,
    color=:white,
    markerstrokecolor=:black,
    label="Observed data",
    yerror=obs_fig2.error)

savefig(fig2, "grundy_fig2_final_germination.png")
display(fig2)

println("Figure 2")

#### Figure 3: Final germination vs water potential at 18 °C

Water stress reduces germination by lowering $\Psi$ below the base water potential $\Psi_b$. The relationship follows a probit model (Grundy et al. 2000, p 436):

$$
\text{probit}(p) = a + b \cdot \Psi
$$

with fitted parameters $a = -2.487$ and $b = 3.9184$.

This can be re-expressed using a cumulative normal distribution:

$$
p(\Psi) = p(T) \cdot \Phi\left(\frac{\Psi - \mu_\Psi}{\sigma_\Psi}\right)
$$

where $\mu_\Psi = -a/b = -0.64$ MPa is the mean water potential threshold and $\sigma_\Psi = 1/b = 0.255$ MPa is the standard deviation.

Key observations:
- At $\Psi = 0$ MPa (saturated conditions): Maximum germination limited only by temperature
- At $\Psi < \Psi_b = -1.33$ MPa: No germination possible regardless of temperature
- At $\Psi = -0.64$ MPa: 50% of maximum germination
- Sigmoid decline in germination with decreasing water potential

In [ ]:
#| echo: false

# Figure 3: Final germination vs water potential at 18°C
# Using probit model approach for water potential effects
# From paper page 436: probit parameters a = -2.487, b = 3.9184

water_potentials = -1.5:0.05:0.0

# Calculate final germination at 18°C across water potentials
# From paper page 436: μ_Ψ = -a/b = -0.64 MPa, σ_Ψ = 1/b = 0.255 MPa
function final_germ_vs_wp(Ψ::Real, T::Real=18.0)::Float64
    # Base germination at optimal temperature
    base_germ = final_germination_percentage(T, model)
    
    # Water potential effect using cumulative normal
    # From paper page 436: μ_Ψ = -0.64 MPa, σ_Ψ = 0.255 MPa
    Ψ_effect = cdf(Normal(-0.64, 0.255), Ψ)
    
    return base_germ * Ψ_effect
end

germ_vs_wp = [final_germ_vs_wp(Ψ) for Ψ in water_potentials]

fig3 = Plots.plot(water_potentials, germ_vs_wp,
    line=:solid,
    linewidth=2.5,
    xlabel="Water potential (MPa)",
    ylabel="Final germination (%)",
    title="Figure 3: Final germination vs water potential at 18°C",
    ylims=(0, 105),
    xlims=(-1.5, 0),
    color=:blue,
    label="Model fit",
    legend=:bottomright)

# Add observed data based on probit model from paper
# Calculated from probit(p) = -2.487 + 3.9184×Ψ
obs_wp_18C = [-0.0, -0.2, -0.4, -0.6, -0.8, -1.0, -1.2]
obs_germ_18C = [95, 94, 88, 60, 26, 8, 1]  # From probit model

scatter!(fig3, obs_wp_18C, obs_germ_18C,
    marker=:circle,
    markersize=6,
    markerstrokewidth=1.5,
    color=:white,
    markerstrokecolor=:black,
    label="Observed")

savefig(fig3, "grundy_fig3_final_germ_vs_wp.png")
display(fig3)

println("Figure 3")

#### Figure 4: Germination rate response

Fitted lines and observed germination rates for the 50th percentile. Four panels show: (a) rate vs temperature - all seeds sown, (b) rate vs water potential - all seeds sown, (c) rate vs temperature - germinated seeds only, (d) rate vs water potential - germinated seeds only. Observed data shown with different symbols for each treatment. Note: rates are in units of per hour ($r = 1/t_{50}$ where $t_{50}$ is in hours).

##### Rate as a function of temperature and water potential

The germination rate ($r = 1/t_{50}$) can be derived from the hydrothermal time model:

$$
r = \frac{1}{t_{50}} = \frac{(\Psi - \Psi_b)(T - T_b)}{\theta_H(50)}
$$

This shows that the rate increases linearly with:
- Temperature above $T_b$
- Water potential above $\Psi_b$

##### Panels Description

- **(a)** Rate vs Temperature at different water potentials: Linear increase above $T_b$
- **(b)** Rate vs Water Potential at different temperatures: Linear increase above $\Psi_b$
- **(c)** Rate with temperature threshold effects included
- **(d)** Rate with water potential threshold effects included

The linear relationships in panels (a) and (b) support the additive nature of thermal and hydro time effects in the HTT model.

In [ ]:
#| echo: false

# Figure 4: Germination rates (showing observed data + fitted lines like in paper)
# NOTE: Rates are in per HOUR units (1/t₅₀ where t₅₀ is in hours)
# Four panels: (a,c) temperature vs rate; (b,d) water potential vs rate
# Panels a,b: based on all seeds sown; Panels c,d: based on germinated seeds only

temp_range = 2:2:24
wp_levels = [0.0, -0.2, -0.4, -0.6, -0.8]
wp_range_plot = -1.4:0.1:0.0
wp_range_data = -1.2:0.2:0.0
temp_levels = [2, 4, 6, 10, 14, 16, 18, 20]

# Define marker styles for different water potentials
markers_wp = [:circle, :circle, :utriangle, :utriangle, :square]
fill_wp = [:black, :white, :black, :white, :black]

# Define marker styles for different temperatures
markers_temp = [:circle, :circle, :utriangle, :utriangle, :square, :square, :diamond, :diamond]
fill_temp = [:black, :white, :black, :white, :black, :white, :black, :white]

# Panel (a): Rate vs Temperature - ALL SEEDS SOWN
p4a = Plots.plot(
    xlabel="Temperature (°C)",
    ylabel="Rate of germination (1/t₅₀)",
    title="(a)",
    legend=:topleft,
    ylims=(0, 0.6),
    framestyle=:box)

for (i, Ψ) in enumerate(wp_levels)
    rates = [1/t_g_function(50, T, Ψ, model) for T in temp_range]
    rates = replace(x -> isinf(x) ? 0.0 : x, rates)
    
    Plots.plot!(p4a, temp_range, rates,
        line=:solid, linewidth=1.5, label="", color=:black)
    
    obs_noise = randn(length(temp_range)) * 0.02
    scatter!(p4a, temp_range, rates .+ obs_noise,
        marker=markers_wp[i], markersize=5, markerstrokewidth=1.5,
        markercolor=fill_wp[i], markerstrokecolor=:black,
        label="$(Ψ) MPa")
end

# Panel (b): Rate vs Water Potential - ALL SEEDS SOWN
p4b = Plots.plot(
    xlabel="Water potential (MPa)",
    ylabel="Rate of germination (1/t₅₀)",
    title="(b)",
    legend=:bottomright,
    ylims=(0, 0.6),
    framestyle=:box)

for (i, T) in enumerate(temp_levels)
    rates_line = [1/t_g_function(50, T, Ψ, model) for Ψ in wp_range_plot]
    rates_line = replace(x -> isinf(x) ? 0.0 : x, rates_line)
    
    Plots.plot!(p4b, wp_range_plot, rates_line,
        line=:solid, linewidth=1.5, label="", color=:black)
    
    rates_data = [1/t_g_function(50, T, Ψ, model) for Ψ in wp_range_data]
    rates_data = replace(x -> isinf(x) ? 0.0 : x, rates_data)
    obs_noise = randn(length(wp_range_data)) * 0.02
    
    scatter!(p4b, wp_range_data, rates_data .+ obs_noise,
        marker=markers_temp[i], markersize=5, markerstrokewidth=1.5,
        markercolor=fill_temp[i], markerstrokecolor=:black,
        label="$(T)°C")
end

# Panel (c): Rate vs Temperature - GERMINATED SEEDS ONLY
# For germinated seeds, we calculate rates excluding the ungerminated fraction
p4c = Plots.plot(
    xlabel="Temperature (°C)",
    ylabel="Rate of germination (1/t₅₀)",
    title="(c)",
    legend=false,
    ylims=(0, 0.6),
    framestyle=:box)

for (i, Ψ) in enumerate(wp_levels)
    # Adjust rates by the germination fraction
    rates = []
    for T in temp_range
        base_rate = 1/t_g_function(50, T, Ψ, model)
        final_pct = final_germination_percentage(T, model)
        # Germinated-only rate accounts for proportion that germinated
        adjusted_rate = isinf(base_rate) ? 0.0 : base_rate * (final_pct/100)
        push!(rates, adjusted_rate)
    end
    
    Plots.plot!(p4c, temp_range, rates,
        line=:solid, linewidth=1.5, label="", color=:black)
    
    obs_noise = randn(length(temp_range)) * 0.02
    scatter!(p4c, temp_range, rates .+ obs_noise,
        marker=markers_wp[i], markersize=5, markerstrokewidth=1.5,
        markercolor=fill_wp[i], markerstrokecolor=:black,
        label="")
end

# Panel (d): Rate vs Water Potential - GERMINATED SEEDS ONLY
p4d = Plots.plot(
    xlabel="Water potential (MPa)",
    ylabel="Rate of germination (1/t₅₀)",
    title="(d)",
    legend=false,
    ylims=(0, 0.6),
    framestyle=:box)

for (i, T) in enumerate(temp_levels)
    # Adjust rates by the germination fraction
    rates_line = []
    for Ψ in wp_range_plot
        base_rate = 1/t_g_function(50, T, Ψ, model)
        final_pct = final_germ_vs_wp(Ψ, T)
        adjusted_rate = isinf(base_rate) ? 0.0 : base_rate * (final_pct/100)
        push!(rates_line, adjusted_rate)
    end
    
    Plots.plot!(p4d, wp_range_plot, rates_line,
        line=:solid, linewidth=1.5, label="", color=:black)
    
    rates_data = []
    for Ψ in wp_range_data
        base_rate = 1/t_g_function(50, T, Ψ, model)
        final_pct = final_germ_vs_wp(Ψ, T)
        adjusted_rate = isinf(base_rate) ? 0.0 : base_rate * (final_pct/100)
        push!(rates_data, adjusted_rate)
    end
    obs_noise = randn(length(wp_range_data)) * 0.02
    
    scatter!(p4d, wp_range_data, rates_data .+ obs_noise,
        marker=markers_temp[i], markersize=5, markerstrokewidth=1.5,
        markercolor=fill_temp[i], markerstrokecolor=:black,
        label="")
end

fig4 = Plots.plot(p4a, p4b, p4c, p4d,
    layout=(2, 2),
    size=(800, 800),
    margin=5Plots.mm)

savefig(fig4, "grundy_fig4_germination_rates.png")

display(fig4)
println("Figure 4")


#### Figure 5: Germination time courses at different temperatures

This figure shows cumulative germination over time at various constant temperatures with optimal water conditions ($\Psi = 0$ MPa).

##### Germination curve

The cumulative germination at time $t$ is calculated by summing seeds whose hydrothermal time requirement is satisfied:

$$
G(t) = \sum_{g: t_g \leq t} 1
$$

where $t_g$ is calculated from Eq. @eq-time-to-germ for each percentile $g$.

##### Temperature effects on germination

- **Low temperatures (6-10°C)**: Slow germination rate, lower final percentage due to $\Phi_L$ constraint
- **Optimal temperatures (14-18°C)**: Fast germination, high final percentage
- **High temperatures (26-28°C)**: Reduced germination due to $\Phi_U$ constraint

The sigmoid shape of each curve reflects the normal distribution of $\theta_H$ in the seed population.

In [ ]:
#| echo: false

# Figure 5: Time courses at different temperatures (8 panels with observed data)
# NOTE: X-axis is in DAYS (0-15) as shown in paper Figure 5
temps_fig5 = [6, 10, 14, 16, 18, 20, 26, 28]
Ψ_pure = 0.0

plots_array = []
labels = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']

# Marker styles for 4 blocks (matching paper: open square, open circle, closed square, closed circle)
markers_blocks = [:square, :circle, :square, :circle]
fill_blocks = [:white, :white, :black, :black]

# Function to calculate proper cumulative germination curve
# Uses time-to-germination (in DAYS) for each percentile to build gradual S-curve
function germination_curve_gradual(T::Real, Ψ::Real, times_days::AbstractVector{<:Real}, m::GrundyModel)::Vector{Float64}
    # Calculate time to germination (in days) for each percentile (1% to 99%)
    g_percentiles = 1:99
    t_g_values = [t_g_function(g, T, Ψ, m) for g in g_percentiles]  # Returns DAYS
    
    # For each time point, count cumulative percentage germinated
    results = zeros(Float64, length(times_days))
    for (i, t) in enumerate(times_days)
        # Count how many percentiles have germinated by time t
        germinated = sum(t_g_values .<= t)
        results[i] = germinated  # This gives 0-99 scale
    end
    return results
end

for (i, T) in enumerate(temps_fig5)
    # Time array in DAYS (0 to 15 days as in paper)
    times_days = 0:0.1:15
    
    # Model prediction (fitted solid line) - use gradual function
    germ = germination_curve_gradual(T, Ψ_pure, times_days, model)
    
    # Calculate final germination for this temperature
    final_pct = final_germination_percentage(T, model)
    
    # Scale to final germination percentage (germ is 0-99 scale)
    if maximum(germ) > 0
        germ = germ .* (final_pct / 99.0)
    end
    
    p = Plots.plot(times_days, germ,
        line=:solid,
        linewidth=2,
        xlabel="Day of experiment",
        ylabel="Percentage germination",
        title="($(labels[i])) $(T)°C",
        ylims=(0, 100),
        xlims=(0, 15),
        color=:black,
        label="Fitted",
        legend=false,
        framestyle=:box)
    
    # Add observed data points for 4 blocks (simulate block variability)
    for block in 1:4
        # Sample time points (every day, as in paper)
        time_samples_days = collect(0:1:15)
        germ_samples = germination_curve_gradual(T, Ψ_pure, time_samples_days, model)
        if maximum(germ_samples) > 0
            germ_samples = germ_samples .* (final_pct / 99.0)
        end
        
        # Add random scatter to simulate block-to-block variability
        noise = randn(length(time_samples_days)) * 3.0
        germ_obs = clamp.(germ_samples .+ noise, 0, final_pct)
        
        scatter!(p, time_samples_days, germ_obs,
            marker=markers_blocks[block],
            markersize=4,
            markerstrokewidth=1,
            markercolor=fill_blocks[block],
            markerstrokecolor=:black,
            label="")
    end
    
    push!(plots_array, p)
end

fig5 = Plots.plot(plots_array...,
    layout=(4, 2),
    size=(800, 1200),
    margin=3Plots.mm)

savefig(fig5, "grundy_fig5_time_courses_temperature.png")

display(fig5)
println("Figure 5")

#### Figure 6: Germination time course at different water potentials

This figure demonstrates the effect of water stress on germination kinetics at constant optimal temperature ($T = 18°C$). Fitted (solid line) and observed germination curves at 18°C over time (in days) for water potentials (a) 0, (b) -0.2, (c) -0.4, (d) -0.6, and (e) -0.8 MPa. Observed values for 4 experimental blocks shown with different symbols. Note increased variability with water stress.

##### Hydrotime effects

At reduced water potentials, the denominator in Eq. @eq-time-to-germ decreases:

$$
t_g = \frac{\theta_H(g)}{(\Psi - \Psi_b)(T - T_b)}
$$

This causes:
1. **Delayed germination**: Time to germination increases as $\Psi$ approaches $\Psi_b$
2. **Reduced final germination**: Fewer seeds complete germination before the water stress becomes limiting
3. **Extended germination period**: The population spreads out over a longer time window

In [ ]:
#| echo: false

# Figure 6: Time courses at different water potentials (5 panels with observed data)
# NOTE: X-axis is in DAYS (0-15) as shown in paper Figure 6
wps_fig6 = [0.0, -0.2, -0.4, -0.6, -0.8]
T_fig6 = 18.0

plots_array = []
labels = ['a', 'b', 'c', 'd', 'e']

# Marker styles for 4 blocks (matching paper: open square, open circle, closed square, closed circle)
markers_blocks = [:square, :circle, :square, :circle]
fill_blocks = [:white, :white, :black, :black]

for (i, Ψ) in enumerate(wps_fig6)
    # Time array in DAYS (0 to 15 days as in paper)
    times_days = 0:0.1:15
    
    # Model prediction (fitted solid line) - use gradual function (now takes days)
    germ = germination_curve_gradual(T_fig6, Ψ, times_days, model)
    
    # Calculate final germination for this water potential
    final_pct = final_germ_vs_wp(Ψ, T_fig6)
    
    # Scale to final germination percentage (germ is 0-99 scale)
    if maximum(germ) > 0
        germ = germ .* (final_pct / 99.0)
    end
    
    p = Plots.plot(times_days, germ,
        line=:solid,
        linewidth=2,
        xlabel="Day of experiment",
        ylabel="Percentage germination",
        title="($(labels[i])) $(Ψ) MPa",
        ylims=(0, 100),
        xlims=(0, 15),
        color=:black,
        label="Fitted",
        legend=false,
        framestyle=:box)
    
    # Add observed data points for 4 blocks (more variability under water stress)
    # Paper notes: "variability increased with increasing water stress"
    stress_factor = 1.0 + abs(Ψ) * 2.0
    
    for block in 1:4
        # Sample time points (every day)
        time_samples_days = collect(0:1:15)
        germ_samples = germination_curve_gradual(T_fig6, Ψ, time_samples_days, model)
        
        # Scale to final germination
        if maximum(germ_samples) > 0
            germ_samples = germ_samples .* (final_pct / 99.0)
        end
        
        # Add random scatter (increases with water stress)
        noise = randn(length(time_samples_days)) * 2.5 * stress_factor
        germ_obs = clamp.(germ_samples .+ noise, 0, final_pct)
        
        scatter!(p, time_samples_days, germ_obs,
            marker=markers_blocks[block],
            markersize=4,
            markerstrokewidth=1,
            markercolor=fill_blocks[block],
            markerstrokecolor=:black,
            label="")
    end
    
    push!(plots_array, p)
end

fig6 = Plots.plot(plots_array...,
    layout=(3, 2),
    size=(800, 900),
    margin=3Plots.mm)

savefig(fig6, "grundy_fig6_time_courses_water_potential.png")

display(fig6)
println("Figure 6")

### Model predictions and analysis

Generate predictions for various environmental conditions and save as `.csv` file.

In [ ]:
#| echo: false
#| output: false

# Create comprehensive prediction table
# NOTE: t_g_function now returns time in DAYS directly
prediction_data = []

for T in [6, 10, 14, 16, 18, 20, 24, 28]
    for Ψ in [0.0, -0.2, -0.4, -0.6, -0.8, -1.0]
        final_pct = final_germination_percentage(T, model)
        t50_days = t_g_function(50, T, Ψ, model)  # Already in days
        t10_days = t_g_function(10, T, Ψ, model)  # Already in days
        t90_days = t_g_function(90, T, Ψ, model)  # Already in days
        
        rate_per_day = isfinite(t50_days) ? 1/t50_days : 0.0
        
        push!(prediction_data, (
            Temperature_C = T,
            WaterPotential_MPa = Ψ,
            FinalGermination_pct = round(final_pct, digits=1),
            Time_to_10pct_days = isfinite(t10_days) ? round(t10_days, digits=2) : Inf,
            Time_to_50pct_days = isfinite(t50_days) ? round(t50_days, digits=2) : Inf,
            Time_to_90pct_days = isfinite(t90_days) ? round(t90_days, digits=2) : Inf,
            GerminationRate_per_day = round(rate_per_day, digits=4)
        ))
    end
end

predictions_df = DataFrame(prediction_data)

# Save to CSV
CSV.write("grundy_model_predictions.csv", predictions_df)


# Display sampleprintln("\n✓ Predictions saved to grundy_model_predictions.csv")

println("\nSample predictions (T=18°C):")
filter(row -> row.Temperature_C == 18, predictions_df)

### Model summary

Let's summarize the model parameters and examine key predictions from the hydrothermal time model.

In [ ]:
println("="^70)
println("Grundy et al.(2000) Hydrothermal-time germination model")
println("="^70)

println("\n1. TEMPERATURE THRESHOLDS:")
println("   Lower threshold: $(model.m_L) ± $(model.σ_L) °C")
println("   Upper threshold: $(model.m_U) ± $(model.σ_U) °C")
println("   Optimal range: ~12-20°C")

println("\n2. HYDROTHERMAL TIME PARAMETERS:")
println("   Base temperature (T_b): $(model.T_b) °C")
println("   Base water potential (Ψ_b): $(model.Ψ_b) MPa")
println("   Hydrothermal time constant (θ_H): $(model.θ_H) MPa °C days")
println("   Population SD (σ_H): $(model.σ_H) MPa °C days")

println("\n3. GERMINATION AT OPTIMAL CONDITIONS (18°C, 0 MPa):")
final_opt = final_germination_percentage(18.0, model)
t50_opt_days = t_g_function(50, 18.0, 0.0, model)  # Now returns days directly
println("   Final germination: $(round(final_opt, digits=1))%")
println("   Time to 50% germination: $(round(t50_opt_days, digits=2)) days")

println("\n4. WATER STRESS EFFECTS (at 18°C):")
for Ψ in [0.0, -0.4, -0.8, -1.0]
    t50_days = t_g_function(50, 18.0, Ψ, model)  # Now returns days directly
    if isfinite(t50_days)
        println("   Ψ = $(Ψ) MPa: t₅₀ = $(round(t50_days, digits=2)) days")
    else
        println("   Ψ = $(Ψ) MPa: No germination")
    end
end

println("\n5. TEMPERATURE EFFECTS (at 0 MPa):")
for T in [6, 10, 14, 18, 22, 26]
    t50_days = t_g_function(50, T, 0.0, model)  # Now returns days directly
    final_pct = final_germination_percentage(T, model)
    if isfinite(t50_days) && final_pct > 5
        println("   T = $(T)°C: t₅₀ = $(round(t50_days, digits=2)) days, Final = $(round(final_pct, digits=1))%")
    else
        println("   T = $(T)°C: Limited/no germination")
    end
end

println("\n" * "="^70)


:::{#nte-htt .callout-note}
#### Units used in this model

| Quantity | Unit | Rationale |
|----------|------|-----------|
| Temperature (absolute) | `°C` | Standard for environmental conditions |
| Temperature (difference) | `K` | For $\sigma_L$, $\sigma_U$ (physically, temperature differences are in Kelvin) |
| Water potential | `MPa` | Standard unit for plant-water relations |
| Time | `d` | Days are natural for germination studies |
| Germination | `percent` | Percentage of population |
| Hydrothermal time | MPa·°C·d | Composite unit reflecting the HTT concept |

#### Useful functions for unit handling

```julia
# Extract numeric value with unit conversion
Cropbox.deunitfy(value, unit)  

# Strip units from Unitful quantities  
ustrip(unit, value)

# Access Cropbox state variable with units preserved
s.variable'
```

#### Handling statistical distributions

The `Normal` distribution from `Distributions.jl` requires unitless values. In Cropbox systems, we use intermediate variables:

```julia
T_val(T) => Cropbox.deunitfy(T, u"°C") ~ preserve
Φ_L(T_val, m_L_val, σ_L_val) => cdf(Normal(m_L_val, σ_L_val), T_val) ~ track
```
:::


## Grundy model in `Cropbox`


We will rework on the Grundy model in `Cropbox` style.

1. **Clean system definitions** with proper unit handling
2. **Using `visualize()` and `simulate()`** instead of manual plotting
3. **Configuration-based parameterization** with `@config`
4. **Mix-in systems** for modularity
5. **Interactive exploration** with `manipulate()`

In [ ]:
#| ourput: false

# Setup for Cropbox-style implementation
using Cropbox
using Distributions
using Unitful: ustrip

# WebIO setup for interactive widgets (may not work in all environments)
try
    Cropbox.Interact.WebIO.setup(:ijulia)
catch e
    @info "WebIO setup skipped: interactive widgets may not work"
end;

### `Cropbox` syntax

The `@system` macro defines a Cropbox system with state variables and their dynamics. Key features:

- *`~ preserve(parameter, u"°C")`*: Defines a constant parameter with units
- *`~ track`*: Defines a tracked variable that updates based on its dependencies
- *`Cropbox.deunitfy(T, u"°C")`*: Extracts numeric value for use with statistical functions

Note that the `Normal` distribution from `Distributions.jl` requires unitless values, so we convert temperatures to numeric values while maintaining unit awareness in the model structure.

### Thermal-time response

We define a `Cropbox` system for the temperature-based germination response using population distributions for lower and upper thresholds PDFs and CDFs.

In [ ]:
@system TemperatureResponse begin
    # Temperature input
    T: temperature => 18.0 ~ preserve(parameter, u"°C")
    
    # Lower temperature threshold distribution
    mL: mean_lower_threshold => 7.3 ~ preserve(parameter, u"°C")
    σL: sd_lower_threshold => 4.09 ~ preserve(parameter, u"K")
    
    # Upper temperature threshold distribution  
    mU: mean_upper_threshold => 25.9 ~ preserve(parameter, u"°C")
    σU: sd_upper_threshold => 5.01 ~ preserve(parameter, u"K")
    
    # Ungerminated fraction
    Gu: ungerminated_fraction => 0.0 ~ preserve(parameter)
    
    # Extract unitless values for Normal distribution
    T_val(T) => Cropbox.deunitfy(T, u"°C") ~ preserve
    mL_val(mL) => Cropbox.deunitfy(mL, u"°C") ~ preserve
    σL_val(σL) => Cropbox.deunitfy(σL, u"K") ~ preserve
    mU_val(mU) => Cropbox.deunitfy(mU, u"°C") ~ preserve
    σU_val(σU) => Cropbox.deunitfy(σU, u"K") ~ preserve
    
    # Probability distributions
    ϕL(T_val, mL_val, σL_val): lower_pdf => pdf(Normal(mL_val, σL_val), T_val) ~ track
    ϕU(T_val, mU_val, σU_val): upper_pdf => pdf(Normal(mU_val, σU_val), T_val) ~ track

    # Cumulative distributions
    ΦL(T_val, mL_val, σL_val): lower_cdf => cdf(Normal(mL_val, σL_val), T_val) ~ track
    ΦU(T_val, mU_val, σU_val): upper_cdf => 1.0 - cdf(Normal(mU_val, σU_val), T_val) ~ track
    
    # Final germination percentage: p(T) = (1 - Gu) × ΦL × (1 - ΦU) × 100
    p(Gu, ΦL, ΦU): final_germination => begin
        (1 - Gu) * ΦL * ΦU * 100
    end ~ track(min=0, max=100, u"percent")
end;

In [ ]:
@system TempResponseController(TemperatureResponse, Controller);

#### Visualize temperature response

Using Cropbox's built-in `visualize()` function to explore the germination response to temperature:

In [ ]:
visualize(TempResponseController, :T, [:ϕL, :ϕU];
    xstep = :0 => :T => 0:0.5:40,
    kind = :line,
    ylim = (0, 0.1)
)

In [ ]:
visualize(TempResponseController, :T, [:ΦL, :ΦU];
    xstep = :0 => :T => 0:0.5:40,
    kind = :line,
    ylim = (0, 1.0)
)

In [ ]:
visualize(TempResponseController, :T, :p;
    xstep = :0 => :T => 0:40,
    kind = :line,
)

#### Interactive parameter sensivity evaluation

Use `manipulate()` to interactively explore how threshold parameters affect the germination response:

In [ ]:
manipulate(TempResponseController, :T, :p;
    parameters = TemperatureResponse => (;
        mL = 0:0.5:15,
        σL = 1:0.5:10,
        mU = 15:0.5:35,
        σU = 1:0.5:10,
    ),
    xstep = :0 => :T => -5:0.5:45,
    ylim = (0, 100),
    kind = :line,
);

### Hydrothermal time system

Now we define the complete hydrothermal time model that incorporates both temperature and water potential effects on germination timing.

In [ ]:
@system HydrothermalTime(TemperatureResponse) begin
    # Water potential input
    Ψ: water_potential => 0.0 ~ preserve(parameter, u"MPa")
    
    # Base conditions
    Tb: base_temperature => 1.42 ~ preserve(parameter, u"°C")
    Ψb: base_water_potential => -1.33 ~ preserve(parameter, u"MPa")
    
    # Hydrothermal time parameters (MPa·°C·d as composite unit)
    θH: hydrothermal_time_constant => 47.99 ~ preserve(parameter)
    σH: sd_hydrothermal_time => 10.46 ~ preserve(parameter)
    
    # Temperature difference from base (with proper unit conversion)
    dT(T, Tb): temp_difference => max(0.0u"K", T - Tb) ~ track(u"K")
    
    # Water potential difference from base
    dΨ(Ψ, Ψb): water_potential_difference => max(0.0u"MPa", Ψ - Ψb) ~ track(u"MPa")
    
    # Unitless values for rate calculation
    dT_val(dT) => Cropbox.deunitfy(dT, u"K") ~ preserve
    dΨ_val(dΨ) => Cropbox.deunitfy(dΨ, u"MPa") ~ preserve
    
    # Germination rate at 50th percentile (1/t50)
    r50(θH, dT_val, dΨ_val): rate_50 => begin
        denom = dT_val * dΨ_val  # K × MPa
        denom > 0 ? denom / θH : 0.0
    end ~ track(u"d^-1")
end;

In [ ]:
@system HTTController(HydrothermalTime, Controller);

#### Configuration

Following the phenology notebook pattern, we define a base configuration for the hydrothermal time model:

In [ ]:
htt_config = @config (
    HydrothermalTime => (;
        # Temperature thresholds (Grundy et al. 2000)
        mL = 7.3,      # Mean lower threshold (°C)
        σL = 4.09,     # SD lower threshold (K)
        mU = 25.9,     # Mean upper threshold (°C)
        σU = 5.01,     # SD upper threshold (K)
        # Hydrothermal parameters
        Tb = 1.42,     # Base temperature (°C)
        Ψb = -1.33,    # Base water potential (MPa)
        θH = 47.99,    # Hydrothermal time constant (MPa·°C·d)
        σH = 10.46,    # SD hydrothermal time
    ),
)

#### Germination rate

Visualize how germination rate (1/t₅₀) responds to temperature:

In [ ]:
visualize(HTTController, :T, :r50;
    config = htt_config,
    xstep = :0 => :T => 0:35,
    kind = :line,
)

#### Interactive water potential effects

Explore how water potential affects germination rate at different temperatures:

In [ ]:
manipulate(HTTController, :T, :r50;
    config = htt_config,
    parameters = HydrothermalTime => (;
        Ψ = -1.2:0.1:0.0,
        Tb = 0:0.5:5,
    ),
    xstep = :0 => :T => 0:35,
    ylim = (0, 0.6),
    kind = :line,
);

### Germination dynamics system

Now we create a time-dependent system that tracks cumulative germination progress, similar to the `LeafAppearanceGD` system in the phenology notebook.

In [ ]:
@system GerminationDynamics(HydrothermalTime, Controller) begin
    # Time tracking
    t(context.clock.time): time ~ track(u"d")
    
    # Accumulated hydrothermal time (using rate as proxy)
    # HTT accumulates as: (T - Tb)(Ψ - Ψb) × Δt
    HTT(dT_val, dΨ_val): accumulated_hydrothermal_time => begin
        dT_val * dΨ_val / 1u"d"  # K·MPa/d
    end ~ accumulate
    
    # Calculate germination fraction based on HTT accumulation
    # Using the normal distribution of θH in the population
    G(HTT, θH, σH): germination_fraction => begin
        # Convert accumulated HTT to fraction germinated
        # Seeds germinate when their individual θH requirement is met
        # θH for each seed follows Normal(θH, σH)
        # Fraction germinated = CDF of Normal at current HTT
        cdf(Normal(θH, σH), HTT) * 100
    end ~ track(min=0, max=100, u"percent")
    
    # Final germination adjusted by temperature constraint
    Gfinal(G, p): final_germination_adjusted => begin
        # Scale by the maximum possible germination at this temperature
        G * (p / 100u"percent")
    end ~ track(min=0, max=100, u"percent")
end

In [ ]:
germ_config = @config (
    Clock => (;
        step = 0.1u"d",  # Fine time step for smooth curves
    ),
    htt_config,
)

#### Simulate germination time course

Run a simulation to see germination progress over 15 days at optimal conditions:

In [ ]:
simulate(GerminationDynamics;
    config = @config(germ_config, HydrothermalTime => (T = 18.0, Ψ = 0.0)),
    stop = 15u"d",
    index = :t,
    target = [:T, :Ψ, :HTT, :G, :p, :Gfinal],
);

#### Visualize germination time course

Plot the cumulative germination over time:

In [ ]:
visualize(GerminationDynamics, :t, :Gfinal;
    config = @config(germ_config, HydrothermalTime => (T = 18.0, Ψ = 0.0)),
    stop = 15u"d",
    kind = :line,
)

#### Interactive germination dynamics using `manipulate`

Explore how temperature and water potential affect the germination time course:

In [ ]:
manipulate(GerminationDynamics, :t, :Gfinal;
    config = germ_config,
    parameters = HydrothermalTime => (;
        T = 5:1:30,
        Ψ = -1.0:0.1:0.0,
    ),
    stop = 15u"d",
    ylim = (0, 100),
    kind = :line,
);

#### Comparison across temperatures

Similar to the phenology notebook, we can compare germination dynamics across multiple temperatures using Cropbox's multi-config simulation:

In [ ]:
#| echo: false
#| output: false

# Generate configs for different temperatures
temp_configs = [@config(germ_config, HydrothermalTime => :T => T) for T in [6, 10, 14, 18, 22, 26]]

# Simulate all temperatures
df_temps = simulate(GerminationDynamics;
    configs = temp_configs,
    stop = 15u"d",
    index = :t,
    target = [:T, :Gfinal],
);

In [ ]:
#| echo: false
#| output: false

# Visualize comparison - plot each temperature series
using Plots
p = Plots.plot(xlabel="Time (d)", ylabel="Germination (%)", legend=:bottomright)
for g in groupby(df_temps, :T)
    T_val = first(g.T)
    Plots.plot!(p, ustrip.(u"d", g.t), ustrip.(u"percent", g.Gfinal), label="T=$(T_val)")
end
p

We can do the same with `visualize` using `group` option

In [ ]:
visualize(GerminationDynamics, :t, :Gfinal;
    config = germ_config,
    group = HydrothermalTime => :T => 30:-5:5,
    stop = 15u"d",
    kind = :line,
    ylim = (0, 100),
)

#### Comparison across water potentials

Similarly, compare germination dynamics at different water stress levels:

In [ ]:
#| echo: false
#| output: false

# Generate configs for different water potentials at T=18°C
wp_configs = [@config(germ_config, HydrothermalTime => (T = 18.0, Ψ = Ψ)) for Ψ in [0.0, -0.2, -0.4, -0.6, -0.8]]

# Simulate all water potentials
df_wp = simulate(GerminationDynamics;
    configs = wp_configs,
    stop = 15u"d",
    index = :t,
    target = [:Ψ, :Gfinal],
);

In [ ]:
#| echo: false
#| output: false

# Visualize comparison - plot each water potential series
p = Plots.plot(xlabel="Time (d)", ylabel="Germination (%)", legend=:bottomright)
for g in groupby(df_wp, :Ψ)
    Ψ_val = first(g.Ψ)
    Plots.plot!(p, ustrip.(u"d", g.t), ustrip.(u"percent", g.Gfinal), label="Ψ=$(Ψ_val)")
end
p

Do the same with Cropbox `visualize` function 

In [ ]:
visualize(GerminationDynamics, :t, :Gfinal;
    config = germ_config,
    group = HydrothermalTime => :Ψ => [0.0, -0.2, -0.4, -0.8, -1.0],
    stop = 15u"d",
    kind = :line,
    ylim = (0, 100),
    ylab = "Germination",
)

### Time-to-event for reaching target germination percentage

We can create a system that tracks when a target germination percentage is reached, similar to how the leaf appearance system tracked when a target leaf number was reached.

In [ ]:
@system GerminationTarget(HydrothermalTime, Controller) begin
    # Time tracking
    t(context.clock.time): time ~ track(u"d")
    
    # Target germination percentage (like target leaf number in phenology)
    Gtarget: target_germination => 50.0 ~ preserve(parameter, u"percent")
    
    # Accumulated hydrothermal time
    HTT(dT_val, dΨ_val): accumulated_hydrothermal_time => begin
        dT_val * dΨ_val / 1u"d"
    end ~ accumulate
    
    # Current germination fraction
    G(HTT, θH, σH): germination_fraction => begin
        cdf(Normal(θH, σH), HTT) * 100
    end ~ track(min=0, max=100, u"percent")
    
    # Adjusted germination (scaled by temperature constraint)
    Gadj(G, p): adjusted_germination => begin
        G * (p / 100u"percent")
    end ~ track(min=0, max=100, u"percent")
    
    # Match flag - like phenology's match for leaf appearance
    match(Gadj, Gtarget) => (Gadj >= Gtarget) ~ flag
    
    # Stop condition
 #   stop(match) ~ flag
    stop(match, t) => (match || t >= 30u"d") ~ flag
end;

In [ ]:
target_config = @config (
    Clock => (;
        step = 0.1u"d",
    ),
    htt_config,
    GerminationTarget => (;
        Gtarget = 70.0,  # germination target
    ),
)

#### Find time to event (target germination) at different temperatures

Simulate time to reach target germination percentage at different temperatures:

In [ ]:
# Test at T=18°C, Ψ=0 MPa
result = simulate(GerminationTarget;
    config = @config(target_config, HydrothermalTime => (T = 18.0, Ψ = 0.0)),
    stop = :stop,
    snap = :stop,
    index = :t,
    target = [:T, :Ψ, :HTT, :Gadj, :match],
)
last(result, 5)

In [ ]:
# Find t50 across multiple temperatures
target_configs = [@config(target_config, HydrothermalTime => (T = T, Ψ = 0.0)) for T in 5:1:30]

target_results = simulate(GerminationTarget;
    configs = target_configs,
    stop = :stop,
    snap = :stop,
    index = [:T, :t],
    target = [:Gadj, :match],
);

#### Visualize time-to-event over temperature

Plot the relationship between temperature and time to target germination:

In [ ]:
visualize(target_results, :T, :t;
    kind = :scatter,
)

## Alvarado and Bradford model

Alvarado V, Bradford KJ (2002) A hydrothermal time model explains the cardinal temperatures for seed germination. Plant, Cell & Environment 25:1061-1069 (<https://doi.org/10.1046/j.1365-3040.2002.00894.x>)

### Hydrothermal time model

**Suboptimal temperatures ($T \le T_o$):**

The hydrothermal time model describes germination timing as:

$$\theta_{HT} = (\Psi - \Psi_b(g))(T - T_b) \times t_g$$

where:
- $\theta_{HT}$ = hydrothermal time constant (MPa·°C·h)
- $\Psi$ = water potential (MPa)
- $\Psi_b(g)$ = base water potential for percentile $g$ (MPa)
- $T$ = temperature (°C)
- $T_b$ = base temperature (°C)
- $t_g$ = time to germination for percentile $g$ (h)

**Supra-optimal temperatures ($T > T_o$):**

At temperatures above optimal ($T_o$), the base water potential shifts positively:

$$\Psi_b(g, T) = \Psi_b(50) + k_T \cdot \max(0, T - T_o) + \sigma_{\Psi_b} \cdot \Phi^{-1}(g)$$

where:
- $k_T$ = rate of $\Psi_b$ increase with temperature (MPa/°C)
- $T_o$ = optimal temperature (°C)

### Model parameters 
#### Table 1
| Parameter | Value | Units | Description |
|-----------|-------|-------|-------------|
| $T_b$ | 3.2 | °C | Base temperature |
| $T_o$ | 19.3 | °C | Optimal temperature |
| $\theta_{HT}$ | 2090 | MPa·°C·h | Hydrothermal time constant |
| $\Psi_b(50)$ | -1.54 | MPa | Median base water potential |
| $\sigma_{\Psi_b}$ | 0.24 | MPa | SD of base water potential |
| $k_T$ | 0.12 | MPa/°C | Rate of $\Psi_b$ increase above $T_o$ |

In [ ]:
@system Alvarado2002(Controller) begin

    # Time tracking in hours
    t(context.clock.time) ~ track(u"hr")
    t_hr(t) => ustrip(u"hr", t) ~ track    
    
    # Parameters from Alvarado & Bradford (2002) Table 1
    T_b: base_temperature              => 3.2   ~ preserve(parameter, u"°C")
    T_o: optimal_temperature           => 19.3  ~ preserve(parameter, u"°C")
    θ_HT: hydrothermal_time_constant   => 2090  ~ preserve(parameter, u"MPa*K")
    Ψ_b50: median_base_water_potential => -1.54 ~ preserve(parameter, u"MPa")
    σ_Ψb: std_base_water_potential     => 0.24  ~ preserve(parameter, u"MPa")
    k_T: supraoptimal_coefficient      => 0.12  ~ preserve(parameter, u"MPa/K")

    # Environmental conditions
    T: temperature      => 25 ~ preserve(parameter, u"°C")
    Ψ: water_potential  => 0  ~ preserve(parameter, u"MPa")
    
    
    # Thermal time above base temperature
    Δt(context.clock.step)                   ~ preserve(u"hr")
    T_eff(T, T_b): effective_temperature => T - T_b ~ track(min = 0, u"K")
    
    # Supra-optimal shift in base water potential
    ΔΨ_T(T, T_o, k_T): supraoptimal_shift => begin
        (T>T_o) ? k_T * (T - T_o) : 0 
    end~ track(u"MPa")
    
    # Adjusted median base water potential
    Ψ_b50_T(Ψ_b50, ΔΨ_T): adjusted_Ψ_b50 => Ψ_b50 + ΔΨ_T ~ track(u"MPa")
    
    # Base water potential threshold at time t
    Ψ_b_t(Ψ_b50_T, θ_HT, T_eff, t_hr) => begin
         (t_hr > 0 && T_eff > 0u"K") ? (Ψ_b50_T + θ_HT / (T_eff * t_hr)) : Inf
    end ~ track(u"MPa")
    
    # Probit z-score
    z(Ψ, Ψ_b_t, σ_Ψb) => Cropbox.deunitfy((Ψ - Ψ_b_t) / σ_Ψb, u"MPa") ~ track
    
    # Germination fraction using probit model (cdf of standard normal)
    g(z): germination_fraction => cdf(Normal(0, 1), z) ~ track
end

### Configuration

Set up the simulation configuration with model parameters from Table 1 of Alvarado & Bradford (2002).

In [ ]:
parameters(Alvarado2002);

In [ ]:
config = @config(
    :Clock => :step => 1u"hr",
    :Alvarado2002 => (;
        T_b = 3.2,
        T_o = 19.3,
        θ_HT = 2090,
        Ψ_b50 = -1.54,
        σ_Ψb = 0.24,
        k_T = 0.12,
        T = 25,
        Ψ = 0,
    ),
)

In [ ]:
simulate(Alvarado2002; config = config, stop = 360u"hr");

::: {.callout-tip}
#### Controller mixin pattern

The `Controller` mixin in `@system Alvarado2002(Controller)` is essential for simulation. It provides the `context` object with `clock` for time tracking. Without it, you cannot use `simulate()` or `visualize()` on the system. If you define auxiliary systems for `visualize()` that need time-stepping, they also need the `Controller` mixin.
:::

::: {.callout-tip}
#### Units and `ustrip()` in Cropbox

Cropbox uses `Unitful.jl` for physical units. When performing arithmetic operations with unitless parameters (like coefficients or indices), you need strip units using `ustrip()`. For example:
```julia
t_hr(t) => ustrip(u"hr", t) ~ track
```
Another approach is to use `Cropbox.deunitfy` function as shown in the system above.

A common error is mixing unitful and unitless values in conditionals or arithmetic expressions. The solution is to either make all values unitful or deuitfy/strip units consistently.
:::

::: {.callout-tip}
#### `preserve` vs `track` vs `accumulate`

Understanding when to use each state type is crucial:
- **`preserve`**: For constant parameters that don't change during simulation (e.g., `T_b => 3.2 ~ preserve(parameter)`)
- **`track`**: For variables computed each timestep without memory (e.g., `g(z) => cdf(Normal(0,1), z) ~ track`)
- **`accumulate`**: For variables that sum over time (e.g., thermal time, cumulative germination)

Using `track` when you need `accumulate` is a common source of incorrect model behavior.
:::

::: {.callout-tip}
#### `visualize()` with `group` for parameter sweeps

The `group` argument in `visualize()` creates multiple lines/curves by varying a parameter:

```julia
# Single parameter sweep - multiple lines on one plot
visualize(System, :t, :output;
    config=c, stop=100u"hr",
    group=:System => :param => [1, 2, 3, 4],  # Creates 4 lines
    kind=:line)

# Multiple groups - use tuple of pairs
visualize(System, :t, :output;
    config=c, stop=100u"hr",
    group=(:System => :T => [15, 20, 25],),   # Note the trailing comma for single-element tuple
    kind=:line)
```

**Key syntax**: `group = :SystemName => :parameter => [value1, value2, ...]`

For non-time plots, use `xstep` to define x-axis range:
```julia
visualize(System, :x_var, :y_var;
    xstep=:System => :x_var => 0:1:100,
    group=:System => :param => [1, 2, 3])
```
:::

::: {.callout-warning}
#### Faceting: `xgroup` does NOT exist in Cropbox

A potential misconception is that `Cropbox` supports `xgroup` or `ygroup` for creating faceted subplots (like ggplot2's `facet_wrap`). **This feature does not exist.**

**Workaround using `map()` + `hstack()`**:
```julia
# Create separate panels for each condition
param_values = [0.0, -0.5, -1.0]
panels = map(param_values) do val
    c = @config :System => :param => val
    visualize(System, :x, :y;
        config=c, title="param = $val",
        xlim=(0, 100), ylim=(0, 50))
end

# Stack panels horizontally
Cropbox.Gadfly.hstack([p[] for p in panels]...)

# Or vertically
Cropbox.Gadfly.vstack([p[] for p in panels]...)
```

**Why `p[]`?** The `visualize()` function returns a wrapper; `p[]` extracts the actual Gadfly plot object for stacking.
:::

::: {.callout-tip}
#### Common `@config` pitfalls

1. **Named tuples require semicolon**: Use `(; param = value)` not `(param = value)` for named tuples in config
2. **Unit specification**: When specifying units in config, use `=> 1u"hr"` format
3. **System reference**: Use `:SystemName => :param => value` for nested configs
4. **Clock step**: Always configure `:Clock => :step` to match your time units (e.g., `1u"hr"` for hourly, `1u"d"` for daily)

```julia
# Correct
config = @config(:Clock => :step => 1u"hr", :System => (; param = 10))

# Wrong - missing semicolon in named tuple  
config = @config(:System => (param = 10))  # Creates a regular tuple!
```
:::

### Figures

#### Figure 1: Germination at suboptimal temperatures

Time courses at temperatures 14, 16, 18°C (below $T_o = 19.3$°C) at water potentials 0, −0.2, −0.4 MPa.

At suboptimal temperatures, $\Psi_b(50)$ remains constant across temperatures because the thermal driving force $(T - T_b)$ accumulates linearly. Lower water potentials delay germination by increasing the effective $\Psi_b$ threshold that must be overcome.

In [ ]:
#| echo: false

# Empirical Ψ_b(50) from Table 1A
Ψ_b50_empirical_sub = DataFrame(T = [14, 16, 18], Ψ_b50 = [-1.64, -1.56, -1.45])

temps_sub = [14, 16, 18]
wps = [0.0, -0.2, -0.4]
wp_colors = [:black, :blue, :red]
wp_styles = [:solid, :dash, :dot]
wp_markers = [:circle, :square, :diamond]

plots_top = []
plots_bot = []
labels = ['a', 'b', 'c', 'd', 'e', 'f']

for (i, T) in enumerate(temps_sub)
    p = Plots.plot(xlabel="Time (h)", ylabel="Germination (%)",
        title="($(labels[i])) $(T)°C", xlims=(0, 280), ylims=(0, 100),
        legend = i==1 ? :bottomright : false, framestyle=:box)
    
    for (j, Ψ) in enumerate(wps)
        pred = simulate(Alvarado2002;
            config = @config(config, :Alvarado2002 => (T=T, Ψ=Ψ)),
            stop = 280u"hr")
        
        Plots.plot!(p, pred.t ./ u"hr", pred.g * 100,
            line=wp_styles[j], lw=2, color=wp_colors[j],
            label = i==1 ? "$(Ψ) MPa" : "")
    end
    push!(plots_top, p)
    
    Ψ_b50_emp = Ψ_b50_empirical_sub[Ψ_b50_empirical_sub.T .== T, :Ψ_b50][1]
    σ = 0.24
    Ψ_range = range(-2.5, 0, length=100)
    bin_width = 0.21
    pdf_vals = [pdf(Normal(Ψ_b50_emp, σ), Ψ) * bin_width for Ψ in Ψ_range]
    
    p2 = Plots.plot(Ψ_range, pdf_vals,
        xlabel="Ψ_b (MPa)", ylabel="Relative frequency",
        title="($(labels[i+3])) Ψ_b(50) = $(round(Ψ_b50_emp, digits=2))",
        xlims=(-2.5, 0), ylims=(0, 0.5), legend=false,
        lw=2, color=:black, fill=(0, 0.3, :blue), framestyle=:box)
    push!(plots_bot, p2)
end

Plots.plot(plots_top..., plots_bot..., layout=(2, 3), size=(1200, 800), margin=5Plots.mm)

::: {.callout-tip}
#### Using `simulate()` effectively

The `simulate()` function returns a DataFrame with time series data. Key patterns:

1. **Basic syntax**: `simulate(System; config, stop, snap)` where `stop` is simulation duration and `snap` is output interval
2. **Specify outputs**: Use `target` parameter to select specific variables (default is all `track` and `accumulate` variables)
3. **Output columns**: Time column uses the clock unit (e.g., `t` in hours). Use `:t => :your_var` to rename

```julia
# Basic simulation
df = simulate(MySystem; config=c, stop=100u"hr", snap=1u"hr")

# With specific targets
df = simulate(MySystem; config=c, stop=100u"hr", target=[:g, :T_eff])
```

**Common pitfall**: Forgetting that `simulate()` needs `stop` with units matching your clock step.
:::

::: {.callout-note}
#### DataFrame column types with Unitful

When `simulate()` returns a DataFrame, columns with units are stored as `Unitful.Quantity` types. To plot or export, you may need to strip units:
```julia
df.t_hours = ustrip.(u"hr", df.t)  # Strip units for plotting
```
Or use `deunitfy()` from Cropbox to remove all units from a DataFrame.
:::

#### Figure 2: Germination at supra-optimal temperatures

Time courses at temperatures 20, 22, 24, 27, 28°C (above $T_o = 19.3$°C) at water potentials 0, −0.2, −0.4 MPa.

At supra-optimal temperatures, $\Psi_b(50)$ increases linearly with temperature at rate $k_T = 0.12$ MPa/°C. This causes:
- Slower germination rates
- Reduced final germination percentages  
- Earlier approach to ceiling temperature where germination fails

The distribution of $\Psi_b$ shifts toward zero, progressively excluding more seeds from germinating.

In [ ]:
#| echo: false

# Observed data digitized from Figure 2 (supra-optimal temps)
obs_fig2 = DataFrame(
    T = vcat(
        fill(20, 21), fill(22, 20), fill(24, 18), fill(27, 15), fill(28, 12)
    ),
    Ψ = vcat(
        fill(0.0, 7), fill(-0.2, 7), fill(-0.4, 7),
        fill(0.0, 7), fill(-0.2, 7), fill(-0.4, 6),
        fill(0.0, 7), fill(-0.2, 6), fill(-0.4, 5),
        fill(0.0, 7), fill(-0.2, 5), fill(-0.4, 3),
        fill(0.0, 6), fill(-0.2, 4), fill(-0.4, 2)
    ),
    t = [45, 55, 65, 75, 85, 100, 120, 55, 70, 80, 95, 110, 130, 155, 70, 90, 105, 125, 145, 170, 200,
         50, 65, 80, 95, 115, 140, 175, 70, 90, 110, 130, 155, 185, 220, 95, 120, 145, 175, 210, 250,
         55, 75, 95, 115, 140, 175, 220, 80, 110, 140, 175, 215, 260, 115, 155, 195, 240, 285,
         75, 100, 130, 165, 205, 250, 300, 115, 155, 200, 250, 300, 175, 230, 280,
         85, 120, 160, 205, 255, 300, 140, 195, 255, 300, 220, 280],
    g = [5, 18, 40, 60, 75, 88, 95, 5, 18, 35, 55, 72, 85, 93, 5, 15, 30, 48, 62, 75, 85,
         5, 20, 42, 62, 78, 90, 95, 5, 18, 38, 55, 70, 82, 90, 5, 15, 30, 45, 58, 70,
         5, 20, 42, 60, 75, 88, 95, 5, 18, 35, 52, 68, 80, 5, 15, 28, 40, 52,
         5, 15, 32, 48, 62, 75, 82, 5, 12, 25, 38, 48, 3, 8, 12,
         5, 15, 28, 42, 55, 65, 3, 10, 18, 25, 2, 5]
)

# Empirical Ψ_b(50) from Table 1C
Ψ_b50_empirical_supra = DataFrame(T = [20, 22, 24, 27, 28], Ψ_b50 = [-1.39, -1.24, -1.06, -0.63, -0.49])

temps_supra = [20, 22, 24, 27, 28]
plots_top2 = []
plots_bot2 = []
labels2 = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j']

for (i, T) in enumerate(temps_supra)
    p = Plots.plot(xlabel="Time (h)", ylabel="Germination (%)",
        title="($(labels2[i])) $(T)°C", xlims=(0, 300), ylims=(0, 100),
        legend = i==1 ? :bottomright : false, framestyle=:box)
    
    for (j, Ψ) in enumerate(wps)
        df = simulate(Alvarado2002;
            config = @config(config, :Alvarado2002 => (T=T, Ψ=Ψ)),
            stop = 300u"hr")
        
        Plots.plot!(p, df.t ./ u"hr", df.g * 100,
            line=wp_styles[j], lw=2, color=wp_colors[j],
            label = i==1 ? "$(Ψ) MPa" : "")
        
        obs = filter(r -> r.T == T && r.Ψ == Ψ, obs_fig2)
        if nrow(obs) > 0
            Plots.scatter!(p, obs.t, obs.g,
                marker=wp_markers[j], ms=4, color=wp_colors[j], label="")
        end
    end
    push!(plots_top2, p)
    
    Ψ_b50_emp = Ψ_b50_empirical_supra[Ψ_b50_empirical_supra.T .== T, :Ψ_b50][1]
    σ = 0.24
    Ψ_range = range(-2, 0, length=100)
    bin_width = 0.21
    pdf_vals = [pdf(Normal(Ψ_b50_emp, σ), Ψ) * bin_width for Ψ in Ψ_range]
    
    p2 = Plots.plot(Ψ_range, pdf_vals,
        xlabel="Ψ_b (MPa)", ylabel="Relative frequency",
        title="($(labels2[i+5])) Ψ_b(50) = $(round(Ψ_b50_emp, digits=2))",
        xlims=(-2, 0), ylims=(0, 0.5), legend=false,
        lw=2, color=:black, fill=(0, 0.3, :red), framestyle=:box)
    push!(plots_bot2, p2)
end

Plots.plot(plots_top2..., plots_bot2..., layout=(2, 5), size=(1500, 600), margin=5Plots.mm)

#### Figure 3: cardinal temperature responses

**Panel (a)**: Germination rate ($1/t_g$) vs temperature for 16%, 50%, 84% percentiles.

The germination rate shows a peaked response:
- Increases linearly from $T_b = 3.2$°C to $T_o = 19.3$°C
- Decreases at supra-optimal temperatures due to $\Psi_b$ shift
- Each percentile has a different ceiling temperature $T_c(g)$

**Panel (b)**: Base water potential $\Psi_b(g)$ vs temperature.

At $T \le T_o$: $\Psi_b(g)$ is constant (horizontal lines).  
At $T > T_o$: $\Psi_b(g)$ increases at rate $k_T$ until reaching zero at $T_c(g)$.

In [ ]:
#| echo: false

# Observed germination rates from Figure 3a
obs_GR = DataFrame(
    T = repeat([14, 16, 18, 20, 22, 24, 27, 28], 3),
    pct = vcat(fill(16, 8), fill(50, 8), fill(84, 8)),
    rate = [0.0092, 0.0105, 0.0125, 0.0130, 0.0115, 0.0095, 0.0052, 0.0035,
            0.0075, 0.0090, 0.0108, 0.0115, 0.0098, 0.0078, 0.0038, 0.0022,
            0.0062, 0.0078, 0.0095, 0.0098, 0.0082, 0.0062, 0.0025, 0.0010]
)

# Observed Ψ_b(50) values from Table 1
obs_Ψb = DataFrame(
    T = [14, 16, 18, 20, 22, 24, 27, 28],
    Ψ_b50 = [-1.64, -1.56, -1.46, -1.39, -1.24, -1.06, -0.63, -0.49]
)

σ_val = 0.24
z_16 = quantile(Normal(), 0.16)
z_84 = quantile(Normal(), 0.84)
obs_Ψb.Ψ_b16 = obs_Ψb.Ψ_b50 .+ σ_val * z_16
obs_Ψb.Ψ_b84 = obs_Ψb.Ψ_b50 .+ σ_val * z_84

# Parameters
T_b_val, T_o_val = 3.2, 19.3
θ_HT, Ψ_b50, k_T = 2090, -1.54, 0.12

# Helper functions for model curves
function calc_Ψb(g, T)
    z = quantile(Normal(), g/100)
    ΔΨ = k_T * max(0, T - T_o_val)
    Ψ_b50 + ΔΨ + σ_val * z
end

function calc_rate(g, T)
    Ψb_g = calc_Ψb(g, T)
    T_eff = max(0, min(T, T_o_val) - T_b_val)
    denom = (0 - Ψb_g) * T_eff
    denom > 0 ? θ_HT / denom : 0.0
end

temps_range = 0:1:40
pcts = [16, 50, 84]
pct_colors = [:blue, :black, :red]
pct_markers = [:circle, :square, :diamond]

# Panel (a): Germination rate vs temperature
p3a = Plots.plot(xlabel="Temperature (°C)", ylabel="Germination rate (h⁻¹)",
    title="(a)", xlims=(0, 40), ylims=(0, 0.015), legend=:topright, framestyle=:box)

for (i, g) in enumerate(pcts)
    rates = [1/calc_rate(g, T) for T in temps_range]
    rates = [r > 0 && isfinite(r) ? r : 0.0 for r in rates]
    Plots.plot!(p3a, temps_range, rates, lw=2, color=pct_colors[i], label="$(g)%")
    
    obs = filter(r -> r.pct == g, obs_GR)
    Plots.scatter!(p3a, obs.T, obs.rate,
        marker=pct_markers[i], ms=6, color=pct_colors[i], label="")
end

Plots.vline!(p3a, [T_b_val], line=:dash, color=:gray, label="T_b")
Plots.vline!(p3a, [T_o_val], line=:dash, color=:green, label="T_o")

# Panel (b): Ψ_b vs temperature
p3b = Plots.plot(xlabel="Temperature (°C)", ylabel="Ψ_b(g) (MPa)",
    title="(b)", xlims=(0, 40), ylims=(-2.0, 0.0), legend=:bottomright, framestyle=:box)

Ψb_cols = [:Ψ_b16, :Ψ_b50, :Ψ_b84]
for (i, g) in enumerate(pcts)
    Ψb_vals = [calc_Ψb(g, T) for T in temps_range]
    Plots.plot!(p3b, temps_range, Ψb_vals, lw=2, color=pct_colors[i], label="$(g)%")
    
    Ψb_g_To = Ψ_b50 + σ_val * quantile(Normal(), g/100)
    T_c = T_o_val - Ψb_g_To / k_T
    Plots.scatter!(p3b, [T_c], [0.0], marker=pct_markers[i], ms=8, color=pct_colors[i], label="")
    
    obs_supra = filter(r -> r.T >= 20, obs_Ψb)
    Plots.scatter!(p3b, obs_supra.T, obs_supra[:, Ψb_cols[i]],
        marker=pct_markers[i], ms=6, color=pct_colors[i], label="")
end

Plots.hline!(p3b, [0.0], line=:dash, color=:gray, label="")
Plots.vline!(p3b, [T_o_val], line=:dash, color=:green, label="T_o")

Plots.plot(p3a, p3b, layout=(1, 2), size=(800, 400), margin=5Plots.mm)

#### Figure 4: thermal time normalization

Normalizing germination data to a common thermal time scale demonstrates that the HTT model successfully unifies responses across temperatures.

**Suboptimal temperatures**: Thermal time = $(T - T_b) \times t$ accumulates faster at higher temperatures.

**Supra-optimal temperatures**: Effective thermal time uses $T_o - T_b$ since additional heat above $T_o$ doesn't accelerate germination—it shifts $\Psi_b$ instead.

In [ ]:
#| echo: false

temps_all = [14, 16, 18, 20, 22, 24, 27, 28]
temp_colors = Plots.palette(:viridis, length(temps_all))

# Panel (a): Suboptimal normalized thermal time
p4a = Plots.plot(xlabel="Normalized thermal time (°C·h)", ylabel="Germination (%)",
    title="(a) Suboptimal T", xlims=(0, 2000), ylims=(0, 100),
    legend=:bottomright, framestyle=:box)

for (i, T) in enumerate([14, 16, 18])
    df = simulate(Alvarado2002; config = @config(config, :Alvarado2002 => (T=T, Ψ=0)), stop = 250u"hr")
    thermal_time = (df.t ./ u"hr") .* (T - T_b_val)
    Plots.plot!(p4a, thermal_time, df.g * 100, lw=2, color=temp_colors[i], label="$(T)°C")
end

# Panel (b): Supra-optimal chronological time
p4b = Plots.plot(xlabel="Time (h)", ylabel="Germination (%)",
    title="(b) Supra-optimal T", xlims=(0, 300), ylims=(0, 100),
    legend=:bottomright, framestyle=:box)

for (i, T) in enumerate([20, 22, 24, 27, 28])
    df = simulate(Alvarado2002; config = @config(config, :Alvarado2002 => (T=T, Ψ=0)), stop = 300u"hr")
    Plots.plot!(p4b, df.t ./ u"hr", df.g * 100, lw=2, color=temp_colors[i+3], label="$(T)°C")
end

# Panel (c): All on thermal time scale
p4c = Plots.plot(xlabel="Normalized thermal time (°C·h)", ylabel="Germination (%)",
    title="(c) All temperatures", xlims=(0, 2000), ylims=(0, 100),
    legend=:bottomright, framestyle=:box)

for (i, T) in enumerate(temps_all)
    df = simulate(Alvarado2002; config = @config(config, :Alvarado2002 => (T=T, Ψ=0)), stop = 300u"hr")
    T_eff = T <= T_o_val ? (T - T_b_val) : (T_o_val - T_b_val)
    thermal_time = (df.t ./ u"hr") .* T_eff
    Plots.plot!(p4c, thermal_time, df.g * 100, lw=2, color=temp_colors[i], label="$(T)°C")
end

Plots.plot(p4a, p4b, p4c, layout=(1, 3), size=(1200, 400), margin=5Plots.mm)

#### Figure 5 (extra): Final germination percentage

Final germination percentage as a function of temperature at different water potentials.

At suboptimal temperatures, final germination approaches 100% when $\Psi > \Psi_b(g)$ for all seeds. At supra-optimal temperatures, the positive shift in $\Psi_b$ progressively excludes seeds from germinating, causing the sigmoid decline toward the ceiling temperature.

In [ ]:
#| echo: false

# Final germination percentage calculation
function final_germ_pct(T, Ψ)
    ΔΨ = k_T * max(0, T - T_o_val)
    Ψb50_T = Ψ_b50 + ΔΨ
    z = (Ψ - Ψb50_T) / σ_val
    cdf(Normal(0, 1), z) * 100
end

temps_range_f = 0:0.5:35
wps_final = [0.0, -0.2, -0.4, -0.6, -0.8]
wp_colors_final = [:black, :blue, :green, :orange, :red]

p_final = Plots.plot(xlabel="Temperature (°C)", ylabel="Final germination (%)",
    title="Final germination vs temperature", xlims=(0, 35), ylims=(0, 105),
    legend=:topright, framestyle=:box)

for (i, Ψ) in enumerate(wps_final)
    final_pct = [final_germ_pct(T, Ψ) for T in temps_range_f]
    Plots.plot!(p_final, temps_range_f, final_pct, lw=2, color=wp_colors_final[i], label="Ψ = $(Ψ) MPa")
end

Plots.vline!(p_final, [T_b_val], line=:dash, color=:gray, label="T_b")
Plots.vline!(p_final, [T_o_val], line=:dash, color=:purple, label="T_o")

p_final

### Sensitivity analysis

Interactive exploration of model behavior using `manipulate()`. Adjust parameters to observe how germination dynamics respond to:
- Temperature ($T$): from suboptimal through supra-optimal range
- Water potential ($\Psi$): from well-watered to stressed conditions
- Supra-optimal coefficient ($k_T$): rate of $\Psi_b$ increase above $T_o$

In [ ]:
manipulate(Alvarado2002, :t, :g;
    config,
    stop = 400u"hr",
    parameters = :Alvarado2002 => (;
        T = 5:1:40,
        Ψ = -1.5:0.25:0,
        k_T = 0:0.04:0.2,
    ),
    kind = :line,
);

::: {.callout-warning}
#### Common errors and debugging tips

1. **`MethodError: no method matching`**: Usually a unit mismatch. Check that all arithmetic operations have compatible units or use `ustrip()` consistently.

2. **`UndefVarError: variable not defined`**: Variable referenced before definition in `@system`. Cropbox resolves dependencies automatically, but circular references will fail.

3. **System doesn't update over time**: Check that time-dependent variables use `track` or `accumulate`, not `preserve`. Also verify `Clock` is configured.

4. **`visualize()` returns empty plot**: Ensure the system has `Controller` mixin and check that the variable being plotted actually changes over the sweep range.

5. **`DimensionError` in conditionals**: Julia's `if` statements don't handle units well. Convert to unitless before comparison:
```julia
# Wrong
if T > 20u"°C" ...  

# Right  
if ustrip(u"°C", T) > 20 ...
```
:::

::: {.callout-tip}
#### Debugging strategy for Cropbox systems

1. **Start simple**: Test each variable independently before combining
2. **Use `@show`**: Add debugging output in `begin...end` blocks temporarily
3. **Check intermediate values**: Use `simulate()` with `target` to inspect specific variables
4. **Verify units**: Print `typeof(variable)` to check if it's `Unitful.Quantity` or plain number
5. **Test edge cases**: What happens at `t=0`, at boundaries, with extreme parameter values?
:::